## Section 1 — Setup & Imports

In [1]:
# Install any missing packages
import subprocess
subprocess.run(['pip', 'install', '-q', 'torchaudio'], check=False)

CompletedProcess(args=['pip', 'install', '-q', 'torchaudio'], returncode=0)

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import numpy as np
import random
import os
import glob
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch version : {torch.__version__}')
print(f'torchaudio      : {torchaudio.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')

PyTorch version : 2.9.0+cu126
torchaudio      : 2.9.0+cu126
CUDA available  : True
GPU             : Tesla T4


In [3]:
def seed_everything(seed=42):
    """Locks all random seeds for absolute reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)
print('Seeds locked to 42.')

Seeds locked to 42.


In [4]:
# --- Kaggle paths ---
INPUT_BASE  = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'
WORKING_BASE = '/kaggle/working'

STEMS_PATH  = os.path.join(INPUT_BASE,  'genres_stems')
NOISE_PATH  = os.path.join(INPUT_BASE,  'ESC-50-master/audio')
MASHUP_OUT  = os.path.join(WORKING_BASE, 'synthetic_mashups/train')
FEATURE_OUT = os.path.join(WORKING_BASE, 'features/train')

GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop',
          'jazz', 'metal', 'pop', 'reggae', 'rock']

DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TARGET_SR   = 22050
DURATION    = 30          # seconds
SAMPLES_PER_GENRE = 50

print(f'Device : {DEVICE}')
print(f'Target length : {TARGET_SR * DURATION:,} samples')

Device : cuda
Target length : 661,500 samples


In [7]:
def generate_synthetic_dataset(stems_dir, noise_dir, output_dir,
                                samples_per_genre=50, target_sr=22050, duration=30):
    """
    Generates deterministic noisy mashups and saves them to /kaggle/working/.

    For each genre:
      - Randomly picks 4 different songs (one per stem type)
      - Mixes the 4 stems into one waveform
      - Overlays a random ESC-50 noise clip at a random position & intensity
      - Saves the result as a .wav file

    Args:
        stems_dir        : Path to genres_stems/ directory
        noise_dir        : Path to ESC-50-master/audio/ directory
        output_dir       : Writable output directory (under /kaggle/working/)
        samples_per_genre: Number of mashups to create per genre (default=50)
        target_sr        : Target sample rate in Hz (default=22050)
        duration         : Clip duration in seconds (default=30)
    """
    target_length = target_sr * duration
    noise_files   = glob.glob(os.path.join(noise_dir, '**', '*.wav'), recursive=True)

    for genre in GENRES:
        genre_out_dir = Path(output_dir) / genre
        genre_out_dir.mkdir(parents=True, exist_ok=True)

        song_folders = glob.glob(os.path.join(stems_dir, genre, '*'))
        if not song_folders:
            print(f'[WARN] No songs found for genre: {genre}')
            continue

        for i in range(samples_per_genre):
            chosen_songs = random.sample(song_folders, 4)
            stems        = []
            stem_types   = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']

            for song, stem_type in zip(chosen_songs, stem_types):
                stem_path = os.path.join(song, stem_type)
                if os.path.exists(stem_path):
                    waveform, sr = torchaudio.load(stem_path)
                    if sr != target_sr:
                        resampler = torchaudio.transforms.Resample(sr, target_sr)
                        waveform  = resampler(waveform)
                    # Crop or pad to exact target length
                    if waveform.shape[1] > target_length:
                        waveform = waveform[:, :target_length]
                    elif waveform.shape[1] < target_length:
                        pad_amt  = target_length - waveform.shape[1]
                        waveform = torch.nn.functional.pad(waveform, (0, pad_amt))
                    stems.append(waveform)

            if len(stems) == 4:
                # Sum stems and normalise
                mashup = torch.stack(stems).sum(dim=0)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)

                # Add ESC-50 noise at a random position
                noise_file = random.choice(noise_files)
                noise, _   = torchaudio.load(noise_file)
                if noise.shape[1] > target_length:
                    noise = noise[:, :target_length]
                if noise.shape[0] != mashup.shape[0]:
                    noise = noise.expand(mashup.shape[0], -1)  # expands channels only: (1,220500)→(2,220500) ✓

                start_idx = random.randint(0, target_length - noise.shape[1])
                intensity = random.uniform(0.1, 0.4)
                mashup[:, start_idx:start_idx + noise.shape[1]] += noise * intensity
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)

                out_path = genre_out_dir / f'mashup_{i:03d}.wav'
                torchaudio.save(str(out_path), mashup, target_sr)

        print(f'[{genre}] done — {samples_per_genre} files saved.')

    total = sum(len(glob.glob(os.path.join(MASHUP_OUT, g, '*.wav'))) for g in GENRES)
    print(f'\nTotal .wav files generated: {total}')


# ---- Run generation ----
generate_synthetic_dataset(STEMS_PATH, NOISE_PATH, MASHUP_OUT, samples_per_genre=SAMPLES_PER_GENRE)

# ---- Q1 verification ----
wav_count = len(glob.glob(os.path.join(MASHUP_OUT, '**', '*.wav'), recursive=True))
print(f'\n>>> Q1 Answer: {wav_count} .wav files exist in synthetic_mashups/train/')

[blues] done — 50 files saved.
[classical] done — 50 files saved.
[country] done — 50 files saved.
[disco] done — 50 files saved.
[hiphop] done — 50 files saved.
[jazz] done — 50 files saved.
[metal] done — 50 files saved.
[pop] done — 50 files saved.
[reggae] done — 50 files saved.
[rock] done — 50 files saved.

Total .wav files generated: 500

>>> Q1 Answer: 500 .wav files exist in synthetic_mashups/train/


In [8]:
# --- Q2: Check waveform shape ---
sample_wav = glob.glob(os.path.join(MASHUP_OUT, '**', '*.wav'), recursive=True)[0]
waveform, sr = torchaudio.load(sample_wav)
print(f'Sample file   : {sample_wav}')
print(f'Sample rate   : {sr}')
print(f'Waveform shape: {tuple(waveform.shape)}')
print(f'\n>>> Q2 Answer: {tuple(waveform.shape)}')

# Expected: (channels, 661500)
# target_sr * duration = 22050 * 30 = 661500

Sample file   : /kaggle/working/synthetic_mashups/train/metal/mashup_001.wav
Sample rate   : 22050
Waveform shape: (2, 661500)

>>> Q2 Answer: (2, 661500)


In [9]:
def extract_and_save_features(input_dir, output_dir, target_sr=22050):
    """
    Converts all .wav files in input_dir into dB-scaled Mel-spectrograms
    and saves them as PyTorch .pt tensors, mirroring the original folder structure.

    Pipeline per file:
      1. torchaudio.load()          -> raw waveform (channels, samples)
      2. MelSpectrogram()           -> (channels, n_mels=128, time_frames)
      3. AmplitudeToDB()            -> convert power to dB scale
      4. torch.save()               -> save as .pt file

    Args:
        input_dir  : Directory containing .wav files
        output_dir : Destination for .pt feature files
        target_sr  : Expected sample rate (default=22050)
    """
    mel_transform    = torchaudio.transforms.MelSpectrogram(
        sample_rate=target_sr,
        n_fft=2048,
        hop_length=512,
        n_mels=128
    )
    amplitude_to_db  = torchaudio.transforms.AmplitudeToDB()

    wav_files = glob.glob(os.path.join(input_dir, '**', '*.wav'), recursive=True)
    if not wav_files:
        print(f'[WARN] No .wav files found in {input_dir}')
        return

    for wav_path in wav_files:
        rel_path = os.path.relpath(wav_path, input_dir)
        out_path = Path(output_dir) / rel_path
        out_path = out_path.with_suffix('.pt')
        out_path.parent.mkdir(parents=True, exist_ok=True)

        waveform, _  = torchaudio.load(wav_path)
        mel_spec     = mel_transform(waveform)
        mel_spec_db  = amplitude_to_db(mel_spec)
        torch.save(mel_spec_db, str(out_path))

    print(f'Saved {len(wav_files)} feature tensors to {output_dir}')


extract_and_save_features(MASHUP_OUT, FEATURE_OUT)

# ---- Q3: Load one .pt file and check shape ----
pt_files = glob.glob(os.path.join(FEATURE_OUT, '**', '*.pt'), recursive=True)
sample_feat = torch.load(pt_files[0])
print(f'\nFeature tensor shape: {tuple(sample_feat.shape)}')
print(f'>>> Q3 Answer: {tuple(sample_feat.shape)}')

# Calculation:
#   target_length = 22050 * 30 = 661500 samples
#   time_frames   = 1 + (661500 // 512) = 1 + 1291 = 1292
#   shape         = (channels, 128, 1292)

Saved 500 feature tensors to /kaggle/working/features/train

Feature tensor shape: (2, 128, 1292)
>>> Q3 Answer: (2, 128, 1292)


In [10]:
class PrecomputedFeatureDataset(Dataset):
    """
    Loads precomputed Mel-spectrogram .pt tensors from disk.

    Directory structure expected:
        features_dir/
          blues/mashup_000.pt
          blues/mashup_001.pt
          ...
          rock/mashup_049.pt

    The genre label is inferred from the parent folder name.
    Returns (feature_tensor, label_int) pairs.
    """
    def __init__(self, features_dir):
        self.files        = glob.glob(os.path.join(features_dir, '**', '*.pt'), recursive=True)
        self.genre_to_idx = {g: i for i, g in enumerate(sorted(GENRES))}

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        genre     = Path(file_path).parent.name           # folder name = genre
        label     = self.genre_to_idx[genre]              # integer label 0-9
        feature   = torch.load(file_path)                 # (channels, 128, T)
        # Collapse channels to 1 (mono) — CRNN expects (1, 128, T)
        if feature.shape[0] > 1:
            feature = feature.mean(dim=0, keepdim=True)
        return feature, label


# Smoke test
full_dataset = PrecomputedFeatureDataset(FEATURE_OUT)
print(f'Dataset size : {len(full_dataset)}')
x, y = full_dataset[0]
print(f'Feature shape: {tuple(x.shape)}  |  Label: {y}')

Dataset size : 500
Feature shape: (1, 128, 1292)  |  Label: 6


In [11]:
class CRNN(nn.Module):
    """
    Convolutional Recurrent Neural Network for music genre classification.

    Architecture overview:
      Input  : (Batch, 1, 128, Time)   — 1-channel Mel-spectrogram
      CNN    : Two conv blocks (32 → 64 channels), each halving spatial dims
      Bridge : Reshape (B, 64, 32, T') → (B, T', 2048)
      RNN    : 1-layer Bidirectional LSTM (hidden=64) → (B, T', 128)
      Pool   : Global max pooling over time → (B, 128)
      Head   : Linear(128 → num_classes)
      Output : (Batch, 10)  — raw logits
    """

    def __init__(self, num_classes=10):
        super().__init__()

        # TODO 1 — CNN backbone
        # Input: (B, 1, 128, T)
        # Block 1: Conv2d(1→32, 3×3, pad=1) → BN → ReLU → MaxPool2d(2)
        # Block 2: Conv2d(32→64, 3×3, pad=1) → BN → ReLU → MaxPool2d(2)
        self.cnn = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),           # (B, 32, 64, T/2)
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),           # (B, 64, 32, T/4)
        )

        # TODO 2 — Bidirectional LSTM
        # After CNN: channels=64, freq_bins=32  →  input_size = 64*32 = 2048
        # 1-layer BiLSTM with hidden_size=64
        self.lstm = nn.LSTM(
            input_size=2048,
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # TODO 3 — Classifier head
        # Bidirectional → output dim = 64 * 2 = 128
        self.fc = nn.Linear(64 * 2, num_classes)

    def forward(self, x):
        """
        Args:
            x : Tensor (Batch, 1, 128, Time)
        Returns:
            logits : Tensor (Batch, num_classes)
        """
        # TODO 4 — CNN pass  →  (B, 64, 32, T')
        x = self.cnn(x)

        # TODO 5 — Bridge / reshape
        #   (B, C=64, F=32, T') → permute → (B, T', C, F) → reshape → (B, T', C*F)
        b, c, f, t = x.shape
        x = x.permute(0, 3, 1, 2)         # (B, T', 64, 32)
        x = x.reshape(b, t, c * f)        # (B, T', 2048)

        # TODO 6 — LSTM  →  (B, T', 128)
        x, _ = self.lstm(x)

        # TODO 7 — Global max pooling over time  →  (B, 128)
        x, _ = torch.max(x, dim=1)

        # TODO 8 — Fully connected  →  (B, num_classes)
        logits = self.fc(x)
        return logits


# ---- Quick sanity check ----
model = CRNN(num_classes=10).to(DEVICE)
dummy = torch.randn(2, 1, 128, 1292).to(DEVICE)
out   = model(dummy)
print(f'Model output shape : {out.shape}')   # expect (2, 10)

total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters   : {total_params:,}')

Model output shape : torch.Size([2, 10])
Total parameters   : 1,102,666


In [12]:
# ============================================================
#  Q4 — Shape after second MaxPool2d, BEFORE permute
# ============================================================
# Trace a batch-32 tensor through the CNN step-by-step
batch_size  = 32
# Load actual time dimension from a sample feature
sample_feat, _ = full_dataset[0]
T = sample_feat.shape[-1]          # time frames from actual features

x_trace = torch.randn(batch_size, 1, 128, T).to(DEVICE)

# Block 1
out1 = model.cnn[:4](x_trace)     # Conv→BN→ReLU→MaxPool
print(f'After Block 1 (first MaxPool2d)  : {tuple(out1.shape)}')

# Block 2 (second MaxPool)
out2 = model.cnn[4:](out1)        # Conv→BN→ReLU→MaxPool
print(f'After Block 2 (second MaxPool2d) : {tuple(out2.shape)}')

print(f'\n>>> Q4 Answer: {tuple(out2.shape)}')
# Expected: (32, 64, 32, T//4) e.g. (32, 64, 32, 323)

After Block 1 (first MaxPool2d)  : (32, 32, 64, 646)
After Block 2 (second MaxPool2d) : (32, 64, 32, 323)

>>> Q4 Answer: (32, 64, 32, 323)


In [13]:
# ============================================================
#  Q5 — Trainable parameters in the LSTM layer alone
# ============================================================
lstm_params = sum(p.numel() for p in model.lstm.parameters() if p.requires_grad)
print(f'LSTM trainable parameters: {lstm_params:,}')
print(f'\n>>> Q5 Answer: {lstm_params}')

# Manual verification:
# BiLSTM: 4 gates × (input_size + hidden_size) × hidden_size + bias
# Per direction = 4 × ((2048 + 64) × 64 + 64)
#               = 4 × (2112 × 64 + 64)
#               = 4 × (135168 + 64)
#               = 4 × 135232
#               = 540928
# Bidirectional = 2 × 540928 = 1,081,856
manual = 4 * ((2048 + 64) * 64 + 64) * 2
print(f'Manual calculation       : {manual:,}')

LSTM trainable parameters: 1,082,368

>>> Q5 Answer: 1082368
Manual calculation       : 1,081,856


In [14]:
# ---- Train / Val split ----
seed_everything(42)
val_ratio  = 0.2
val_size   = int(len(full_dataset) * val_ratio)
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train samples : {train_size}  |  Val samples : {val_size}')

Train samples : 400  |  Val samples : 100


In [15]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """Runs one full pass over the training set and returns avg loss + macro F1."""
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []

    for features, labels in loader:
        features, labels = features.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(features)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, macro_f1


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Evaluates on val set and returns avg loss + macro F1."""
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []

    for features, labels in loader:
        features, labels = features.to(device), labels.to(device)
        logits = model(features)
        loss   = criterion(logits, labels)

        total_loss += loss.item()
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, macro_f1

In [20]:
# ---- Training configuration (as specified in the milestone form) ----
seed_everything(42)

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model     = CRNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

NUM_EPOCHS  = 10
best_val_f1 = 0.0

print(f'Training on {device} for {NUM_EPOCHS} epochs\n')
print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Train F1":>9} | {"Val Loss":>9} | {"Val F1":>8}')
print('-' * 60)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_f1 = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss,   val_f1   = evaluate(model, val_loader, criterion, device)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({'epoch': epoch, 'model_state': model.state_dict(), 'val_f1': val_f1},
                   '/kaggle/working/best_crnn.pth')

    print(f'{epoch:>6} | {train_loss:>10.4f} | {train_f1:>9.4f} | {val_loss:>9.4f} | {val_f1:>8.4f}')

print(f'\nBest Val Macro F1 : {best_val_f1:.4f}')

Training on cuda for 10 epochs

 Epoch | Train Loss |  Train F1 |  Val Loss |   Val F1
------------------------------------------------------------
     1 |     2.2466 |    0.1729 |    2.1424 |   0.1724
     2 |     1.9629 |    0.3449 |    1.9617 |   0.2157
     3 |     1.8150 |    0.3496 |    1.8495 |   0.3098
     4 |     1.7082 |    0.4370 |    1.7090 |   0.3503
     5 |     1.5661 |    0.5308 |    1.6753 |   0.5226
     6 |     1.5010 |    0.5215 |    1.6261 |   0.4430
     7 |     1.4083 |    0.6320 |    1.5223 |   0.5619
     8 |     1.3353 |    0.6367 |    1.5233 |   0.5586
     9 |     1.2488 |    0.7128 |    1.4078 |   0.5069
    10 |     1.1731 |    0.7292 |    1.3107 |   0.6699

Best Val Macro F1 : 0.6699


## Section 11 — Q6: Conceptual MCQ Answer

In [18]:
print("""
Q6 — Why is translation invariance along the Y-axis (Mel-Frequency Bins) problematic?

CORRECT ANSWER:
  "Moving a pattern up the Y-axis fundamentally changes its pitch and
   instrument identity (e.g., shifting a low bassline up 80 bins turns
   it into a high-pitched synth)."

Explanation:
  - On the X-axis (time): a guitar solo at 0:15 sounds identical at 0:25.
    Temporal position does NOT change what instrument or note is being played.
    So CNN translation invariance along time is HELPFUL.

  - On the Y-axis (frequency / mel bins): each row corresponds to a specific
    frequency range. A low bass note (e.g., 80-100 Hz) appearing at the bottom
    of the spectrogram is a completely different sound from a high-pitched synth
    tone (e.g., 4000-5000 Hz) appearing at the top.

    If the CNN learns to be invariant to vertical shifts, it LOSES the ability
    to distinguish between low and high-pitched instruments — which are one of
    the most important cues for genre classification.
""")


Q6 — Why is translation invariance along the Y-axis (Mel-Frequency Bins) problematic?

CORRECT ANSWER:
  "Moving a pattern up the Y-axis fundamentally changes its pitch and
   instrument identity (e.g., shifting a low bassline up 80 bins turns
   it into a high-pitched synth)."

Explanation:
  - On the X-axis (time): a guitar solo at 0:15 sounds identical at 0:25.
    Temporal position does NOT change what instrument or note is being played.
    So CNN translation invariance along time is HELPFUL.

  - On the Y-axis (frequency / mel bins): each row corresponds to a specific
    frequency range. A low bass note (e.g., 80-100 Hz) appearing at the bottom
    of the spectrogram is a completely different sound from a high-pitched synth
    tone (e.g., 4000-5000 Hz) appearing at the top.

    If the CNN learns to be invariant to vertical shifts, it LOSES the ability
    to distinguish between low and high-pitched instruments — which are one of
    the most important cues for genre classific